# BoltzGen Inference (Processing Job) on SageMaker

이 노트북에서는 SageMaker Processing Job을 사용하여 BoltzGen으로 단백질 바인더를 설계(추론)하는 방법을 실습합니다.

## BoltzGen 설계 파이프라인

```
Design Spec (YAML) → Design (Diffusion) → Inverse Folding → Folding (Boltz-2) → Analysis → Filtering → 최종 바인더
```

## 실습 순서

1. 환경 설정
2. 설계 사양(Design Spec) 준비
3. SageMaker Processing Job 실행
4. 결과 모니터링
5. 결과 다운로드 및 분석

## 1. 환경 설정

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

# SageMaker 세션 및 역할
sagemaker_session = sagemaker.Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"Role: {role}")

In [ ]:
# S3 버킷 설정
S3_BUCKET = sagemaker_session.default_bucket()
S3_PREFIX = "boltzgen"

# 사전 빌드된 BoltzGen 이미지 (Cross-account ECR)
# 워크샵용으로 공개 설정된 이미지입니다.
BOLTZGEN_IMAGE_ACCOUNT = "418272795925"
REPOSITORY_NAME = "boltzgen-sagemaker"
IMAGE_TAG = "latest"

# SageMaker Processing Job은 같은 리전의 ECR만 사용 가능
IMAGE_URI = f"{BOLTZGEN_IMAGE_ACCOUNT}.dkr.ecr.{region}.amazonaws.com/{REPOSITORY_NAME}:{IMAGE_TAG}"

print(f"S3 Bucket: {S3_BUCKET}")
print(f"Image URI: {IMAGE_URI}")

## 2. 설계 사양(Design Spec) 준비

BoltzGen은 YAML 형식의 설계 사양(Design Spec)을 입력으로 받습니다.

### 지원 프로토콜

| 프로토콜 | 용도 |
|---------|------|
| `protein-anything` | 단백질 바인더 설계 (기본) |
| `peptide-anything` | 펩타이드 바인더 설계 |
| `protein-small_molecule` | 소분자 결합 단백질 설계 |
| `nanobody-anything` | 나노바디 설계 |

In [ ]:
# 설계 사양 파일 확인
design_spec_path = Path("examples/vanilla_protein.yaml")
print(f"Design spec: {design_spec_path}")
print(f"\n--- {design_spec_path} ---")
print(design_spec_path.read_text())

In [ ]:
# 설계 사양 및 관련 파일을 S3에 업로드
s3_client = boto3.client('s3')

design_spec_filename = design_spec_path.name
s3_input_prefix = f"{S3_PREFIX}/input"

# YAML 파일 업로드
s3_key = f"{s3_input_prefix}/{design_spec_filename}"
s3_client.upload_file(str(design_spec_path), S3_BUCKET, s3_key)
print(f"Uploaded: s3://{S3_BUCKET}/{s3_key}")

# 같은 디렉토리에 있는 .cif/.pdb 파일도 업로드
design_dir = design_spec_path.parent
for ext in ['*.cif', '*.pdb']:
    for file in design_dir.glob(ext):
        s3_key = f"{s3_input_prefix}/{file.name}"
        s3_client.upload_file(str(file), S3_BUCKET, s3_key)
        print(f"Uploaded: s3://{S3_BUCKET}/{s3_key}")

input_s3_uri = f"s3://{S3_BUCKET}/{s3_input_prefix}"
print(f"\nInput S3 URI: {input_s3_uri}")

## 3. SageMaker Processing Job 실행

### 인스턴스 권장 사양

| 인스턴스 | GPU | 메모리 | 용도 | 비용 (시간당) |
|----------|-----|--------|------|---------------|
| ml.g4dn.xlarge | 1 | 16GB | 소규모 테스트 | ~$0.74 |
| ml.g5.xlarge | 1 | 24GB | 일반 추론 | ~$1.41 |
| ml.g5.2xlarge | 1 | 32GB | 대규모 설계 | ~$1.63 |

In [ ]:
# Processing Job 설정
INSTANCE_TYPE = "ml.g5.2xlarge"     # GPU 인스턴스
VOLUME_SIZE = 50                    # EBS 볼륨 크기 (GB)
MAX_RUNTIME = 86400                 # 최대 실행 시간 (초, 24시간)

# BoltzGen 설계 파라미터
PROTOCOL = "protein-anything"       # 설계 프로토콜
NUM_DESIGNS = 10                    # 생성할 설계 수
BUDGET = 2                          # 최종 다양성 최적화 설계 수

In [ ]:
# ScriptProcessor 생성
processor = ScriptProcessor(
    role=role,
    image_uri=IMAGE_URI,
    instance_count=1,
    instance_type=INSTANCE_TYPE,
    volume_size_in_gb=VOLUME_SIZE,
    max_runtime_in_seconds=MAX_RUNTIME,
    base_job_name='boltzgen',
    command=['python3'],
)

print("ScriptProcessor created successfully!")

In [ ]:
# Job 이름 및 출력 경로 설정
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
job_name = f"boltzgen-{timestamp}"
output_s3_uri = f"s3://{S3_BUCKET}/{S3_PREFIX}/output/{job_name}"

print(f"Job name: {job_name}")
print(f"Input: {input_s3_uri}")
print(f"Output: {output_s3_uri}")
print(f"Protocol: {PROTOCOL}")
print(f"Num designs: {NUM_DESIGNS}")
print(f"Budget: {BUDGET}")

In [ ]:
# Processing Job 실행
processor.run(
    code='scripts/processing_script.py',
    inputs=[
        ProcessingInput(
            source=input_s3_uri,
            destination='/opt/ml/processing/input',
            input_name='design-spec'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=output_s3_uri,
            output_name='results'
        )
    ],
    arguments=[
        '--design-spec', design_spec_filename,
        '--protocol', PROTOCOL,
        '--num-designs', str(NUM_DESIGNS),
        '--budget', str(BUDGET),
        '--devices', '1'
    ],
    wait=False,  # 비동기 실행
    logs=True,
    job_name=job_name
)

print(f"\nProcessing Job launched: {job_name}")

## 4. 결과 모니터링

In [ ]:
# Processing Job 상태 확인
sm_client = boto3.client('sagemaker')

response = sm_client.describe_processing_job(ProcessingJobName=job_name)
status = response['ProcessingJobStatus']
print(f"Job Name: {job_name}")
print(f"Status: {status}")

if status == 'Completed':
    duration = response['ProcessingEndTime'] - response['ProcessingStartTime']
    print(f"Duration: {duration}")
elif status == 'Failed':
    print(f"Failure Reason: {response.get('FailureReason', 'N/A')}")

In [ ]:
# CloudWatch 로그 확인
!aws logs tail /aws/sagemaker/ProcessingJobs --log-stream-name-prefix {job_name} --limit 20

In [ ]:
# (선택) 작업 완료까지 대기
# import time
# while True:
#     response = sm_client.describe_processing_job(ProcessingJobName=job_name)
#     status = response['ProcessingJobStatus']
#     print(f"Status: {status}")
#     if status in ['Completed', 'Failed', 'Stopped']:
#         break
#     time.sleep(30)

## 5. 결과 다운로드 및 분석

In [ ]:
# S3 결과 목록 확인
!aws s3 ls {output_s3_uri}/ --recursive --human-readable

In [ ]:
# 결과 다운로드
local_output_dir = Path(f"results/{job_name}")
local_output_dir.mkdir(parents=True, exist_ok=True)

!aws s3 sync {output_s3_uri} {local_output_dir}

print(f"\nResults downloaded to: {local_output_dir}")

In [ ]:
# Job 메타데이터 확인
metadata_file = local_output_dir / "job_metadata.json"
if metadata_file.exists():
    with open(metadata_file) as f:
        metadata = json.load(f)
    print("Job Metadata:")
    print(json.dumps(metadata, indent=2))
else:
    print("Metadata file not found (job may still be running)")

In [ ]:
# 생성된 결과 파일 목록
results_dir = local_output_dir / "results"
if results_dir.exists():
    for f in sorted(results_dir.rglob("*")):
        if f.is_file():
            size_kb = f.stat().st_size / 1024
            print(f"  {f.relative_to(local_output_dir)}: {size_kb:.1f} KB")
else:
    print("Results directory not found")

## 정리

이 노트북에서는 다음을 학습했습니다:

- BoltzGen 설계 사양(Design Spec) YAML 형식 이해
- SageMaker `ScriptProcessor`를 사용한 Processing Job 구성
- 단백질 바인더 설계 실행 및 결과 분석

### 추가 실습 아이디어

- `NUM_DESIGNS`를 100으로 늘려서 더 많은 바인더 후보 생성
- `peptide-anything` 프로토콜로 펩타이드 바인더 설계
- `nanobody-anything` 프로토콜로 나노바디 설계
- 더 큰 인스턴스(`ml.g5.12xlarge`)로 대규모 배치 실행